# MEDS Output Review: MIMIC + AHS

This notebook reviews the finalized flat patient-bucketed MEDS outputs.
It avoids loading the full datasets by default. The first sections read manifests, patient indexes,
and parquet footers; heavier validation and distribution scans are opt-in cells.

Target final layout:

```text
{source}_meds_core_by_patient/
  bucket000.parquet
  ...
  bucket255.parquet
{source}_patient_index.parquet
```

Expected finalized component order for both sources:
`ecgs, labs, admissions, eds, diagnosis, procedures, medicines, demographics`.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

OUTPUT_ROOT = Path("/data/padmalab_external/special_project/meds_pipeline_output")
SOURCES = {
    "mimic": OUTPUT_ROOT / "mimic",
    "ahs": OUTPUT_ROOT / "ahs",
}
EXPECTED_COMPONENTS = [
    "ecgs",
    "labs",
    "admissions",
    "eds",
    "diagnosis",
    "procedures",
    "medicines",
    "demographics",
]
STABLE_STRING_COLUMNS = [
    "subject_id",
    "event_type",
    "code",
    "code_system",
    "value_num",
    "value_text",
    "unit",
    "comparator",
    "source_table",
    "provenance_id",
    "encounter_id",
    "site",
    "ecg_id",
]
SORT_COLUMNS = ["subject_id", "time", "event_type", "code"]


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "src" / "meds_pipeline").exists():
            return path
    return start


REPO_ROOT = find_repo_root()
VALIDATOR_SCRIPT = REPO_ROOT / "scripts" / "validate_patient_bucketed_output.py"
print("repo_root:", REPO_ROOT)
print("output_root:", OUTPUT_ROOT)
print("validator_script:", VALIDATOR_SCRIPT, "exists=", VALIDATOR_SCRIPT.exists())

## Path and Manifest Helpers

These helpers point at the flat finalized bucket files and the patient index table.

In [ ]:
def source_output_dir(source: str) -> Path:
    return SOURCES[source]


def manifest_path(source: str) -> Path:
    return source_output_dir(source) / "_staging_meds_core_by_patient" / "manifest.json"


def dataset_dir(source: str) -> Path:
    return source_output_dir(source) / f"{source}_meds_core_by_patient"


def patient_index_path(source: str) -> Path:
    return source_output_dir(source) / f"{source}_patient_index.parquet"


def bucket_file_path(source: str, bucket: str) -> Path:
    return dataset_dir(source) / f"bucket{bucket}.parquet"


def load_manifest(source: str) -> dict:
    path = manifest_path(source)
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))


def bucket_files(source: str) -> list[Path]:
    return sorted(dataset_dir(source).glob("bucket*.parquet"))


def patient_index(source: str) -> pd.DataFrame:
    path = patient_index_path(source)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_parquet(path)


def component_rows(source: str) -> pd.DataFrame:
    manifest = load_manifest(source)
    rows = [
        {
            "source": source,
            "order": idx,
            "component": component["name"],
            "row_count": int(component["row_count"]),
            "bucket_count": len(component.get("buckets", {})),
            "parts": component.get("parts"),
        }
        for idx, component in enumerate(manifest.get("components", []), start=1)
    ]
    return pd.DataFrame(rows)


def manifest_summary(source: str) -> dict:
    manifest = load_manifest(source)
    components = [component["name"] for component in manifest.get("components", [])]
    rows = sum(int(component["row_count"]) for component in manifest.get("components", []))
    return {
        "source": source,
        "manifest_exists": manifest_path(source).exists(),
        "dataset_exists": dataset_dir(source).exists(),
        "patient_index_exists": patient_index_path(source).exists(),
        "num_buckets": int(manifest.get("num_buckets", -1)),
        "component_order_ok": components == EXPECTED_COMPONENTS,
        "manifest_components": ", ".join(components),
        "manifest_row_sum": rows,
        "patient_ids": manifest.get("patient_ids"),
    }

## Fast Status Overview

This reads only the staging manifest, patient index, and parquet footers. It is safe to rerun frequently.

In [ ]:
def footer_summary(source: str) -> dict:
    files = bucket_files(source)
    total_rows = 0
    first_schema = None
    schemas_equal = True
    schema_mismatch_count = 0

    for path in files:
        schema = pq.read_schema(path).remove_metadata()
        if first_schema is None:
            first_schema = schema
        elif schema != first_schema:
            schemas_equal = False
            schema_mismatch_count += 1
        total_rows += pq.ParquetFile(path).metadata.num_rows

    schema_types = {field.name: field.type for field in first_schema} if first_schema else {}
    type_mismatches = []
    expected_types = {"time": pa.timestamp("ns")}
    expected_types.update({column: pa.string() for column in STABLE_STRING_COLUMNS})
    for column, expected in expected_types.items():
        actual = schema_types.get(column)
        if actual != expected:
            type_mismatches.append({"column": column, "expected": str(expected), "actual": str(actual)})

    manifest = load_manifest(source)
    manifest_rows = sum(int(component["row_count"]) for component in manifest.get("components", []))
    idx = patient_index(source) if patient_index_path(source).exists() else pd.DataFrame()
    return {
        "source": source,
        "file_count": len(files),
        "total_rows_from_footers": total_rows,
        "manifest_row_sum": manifest_rows,
        "rows_match_manifest": total_rows == manifest_rows,
        "schemas_equal": schemas_equal,
        "schema_mismatch_count": schema_mismatch_count,
        "type_mismatch_count": len(type_mismatches),
        "type_mismatches": type_mismatches,
        "time_type": str(schema_types.get("time")),
        "patient_index_rows": len(idx),
        "patient_index_row_sum": int(idx["row_count"].sum()) if len(idx) else 0,
        "patient_index_row_sum_matches": int(idx["row_count"].sum()) == total_rows if len(idx) else False,
    }


status = pd.DataFrame([{**manifest_summary(source), **footer_summary(source)} for source in SOURCES])
status

In [ ]:
component_table = pd.concat([component_rows(source) for source in SOURCES], ignore_index=True)
component_table

In [ ]:
component_pivot = component_table.pivot(index="component", columns="source", values="row_count").loc[EXPECTED_COMPONENTS]
component_pivot["mimic_minus_ahs"] = component_pivot["mimic"] - component_pivot["ahs"]
component_pivot

In [ ]:
pd.concat(
    [patient_index(source).assign(source=source).head(10) for source in SOURCES],
    ignore_index=True,
)

## Full Invariant Validation

This calls `scripts/validate_patient_bucketed_output.py`. It scans every final bucket and checks row totals, schema equality,
stable column types, patient-to-bucket confinement, bucket sort order, and patient index consistency. It can take a few minutes per source.

In [ ]:
def run_validator(source: str) -> subprocess.CompletedProcess[str]:
    if not VALIDATOR_SCRIPT.exists():
        raise FileNotFoundError(VALIDATOR_SCRIPT)
    cmd = [
        sys.executable,
        str(VALIDATOR_SCRIPT),
        "--output-dir",
        str(source_output_dir(source)),
        "--expected-components",
        ",".join(EXPECTED_COMPONENTS),
        "--progress",
    ]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, text=True, check=False)


# Uncomment one or both when you want a full revalidation from inside the notebook.
# run_validator("mimic")
# run_validator("ahs")

## Schema Review

Use this to inspect the exact schema written in a final bucket file. All final bucket files should share this schema.

In [ ]:
def schema_frame(source: str, bucket: str = "000") -> pd.DataFrame:
    path = bucket_file_path(source, bucket)
    schema = pq.read_schema(path).remove_metadata()
    return pd.DataFrame(
        {
            "field": schema.names,
            "type": [str(field.type) for field in schema],
        }
    )


schema_frame("mimic").merge(
    schema_frame("ahs"),
    on="field",
    how="outer",
    suffixes=("_mimic", "_ahs"),
)

## Lightweight Row Sampling

These cells read one bucket file at a time. They are intended for visual review and examples, not global statistics.

In [ ]:
def read_bucket_head(
    source: str,
    bucket: str = "000",
    n: int = 20,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    table = pq.read_table(bucket_file_path(source, bucket), columns=columns)
    return table.slice(0, n).to_pandas()


review_columns = [
    "subject_id",
    "time",
    "event_type",
    "code",
    "code_system",
    "value_num",
    "value_text",
    "unit",
    "source_table",
    "provenance_id",
    "encounter_id",
    "site",
    "ecg_id",
]

read_bucket_head("mimic", bucket="000", n=20, columns=review_columns)

In [ ]:
read_bucket_head("ahs", bucket="000", n=20, columns=review_columns)

## Event and Code Distribution on Selected Buckets

By default this scans four buckets per source. Increase `buckets` only when you are comfortable with the extra read time.

In [ ]:
def selected_bucket_paths(source: str, buckets: list[str] | tuple[str, ...]) -> list[Path]:
    paths = []
    for bucket in buckets:
        path = bucket_file_path(source, bucket)
        if path.exists():
            paths.append(path)
        else:
            print("missing", path)
    return paths


def value_counts_for_column(
    source: str,
    column: str,
    buckets: tuple[str, ...] = ("000", "001", "002", "003"),
    top_n: int = 30,
) -> pd.DataFrame:
    counts = pd.Series(dtype="int64")
    rows_scanned = 0
    for path in selected_bucket_paths(source, buckets):
        s = pq.read_table(path, columns=[column]).to_pandas()[column]
        rows_scanned += len(s)
        counts = counts.add(s.astype("string").fillna("<NA>").value_counts(), fill_value=0)
    out = counts.sort_values(ascending=False).head(top_n).astype("int64").rename("count").reset_index()
    out = out.rename(columns={"index": column})
    out.insert(0, "source", source)
    out.insert(1, "rows_scanned", rows_scanned)
    return out


pd.concat(
    [value_counts_for_column(source, "event_type") for source in SOURCES],
    ignore_index=True,
)

In [ ]:
def code_prefix_counts(
    source: str,
    buckets: tuple[str, ...] = ("000", "001", "002", "003"),
    levels: int = 2,
    top_n: int = 30,
) -> pd.DataFrame:
    counts = pd.Series(dtype="int64")
    rows_scanned = 0
    for path in selected_bucket_paths(source, buckets):
        codes = pq.read_table(path, columns=["code"]).to_pandas()["code"].astype("string")
        rows_scanned += len(codes)
        prefixes = codes.str.split("//").str[:levels].str.join("//").fillna("<NA>")
        counts = counts.add(prefixes.value_counts(), fill_value=0)
    out = counts.sort_values(ascending=False).head(top_n).astype("int64").rename("count").reset_index()
    out = out.rename(columns={"index": f"code_prefix_{levels}"})
    out.insert(0, "source", source)
    out.insert(1, "rows_scanned", rows_scanned)
    return out


pd.concat([code_prefix_counts(source, levels=2) for source in SOURCES], ignore_index=True)

## Patient Timeline Review

Use the patient index to locate the right bucket file for a `subject_id`, then read only that file.

In [ ]:
def locate_patient(source: str, subject_id: str) -> pd.Series:
    idx = patient_index(source)
    match = idx[idx["subject_id"].astype(str) == str(subject_id)]
    if match.empty:
        raise KeyError(f"{subject_id!r} not found in {source} patient index")
    if len(match) != 1:
        raise ValueError(f"{subject_id!r} has {len(match)} patient index rows")
    return match.iloc[0]


def read_patient(
    source: str,
    subject_id: str,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    row = locate_patient(source, subject_id)
    path = source_output_dir(source) / str(row["file_path"])
    columns = columns or review_columns
    df = pq.read_table(path, columns=columns).to_pandas()
    patient = df[df["subject_id"].astype(str) == str(subject_id)].copy()
    return patient.sort_values(SORT_COLUMNS, kind="mergesort", na_position="last").reset_index(drop=True)


def first_subject_id(source: str) -> str:
    idx = patient_index(source)
    if idx.empty:
        raise ValueError(f"No patients in {source} patient index")
    return str(idx.iloc[0]["subject_id"])


def patient_timeline(source: str, subject_id: str | None = None, n: int = 100) -> pd.DataFrame:
    subject_id = subject_id or first_subject_id(source)
    return read_patient(source, subject_id).head(n)


patient_timeline("mimic", n=100)

In [ ]:
patient_timeline("ahs", n=100)

## Optional Label File Spot Check

The old notebook looked at an AHS HF readmission label parquet. This section keeps that check optional and isolated from the core output review.

In [ ]:
LABEL_PATHS = {
    "ahs_hf_readmission_excluded": Path(
        "/data/padmalab_external/special_project/Weijie_Code/MEDs_label/ahs_discharged_hf_readmission_excluded.parquet"
    ),
}


def preview_label(name: str, n: int = 20) -> pd.DataFrame:
    path = LABEL_PATHS[name]
    if not path.exists():
        raise FileNotFoundError(path)
    return pq.read_table(path).slice(0, n).to_pandas()


# preview_label("ahs_hf_readmission_excluded")

## Export a Small Review Sample

This writes small shuffled samples to `tests/meds_core_head_{source}.csv` for quick external inspection.

In [ ]:
def export_small_sample(
    source: str,
    bucket: str = "000",
    n: int = 100,
    random_state: int = 42,
) -> Path:
    df = read_bucket_head(source, bucket=bucket, n=max(n * 10, n), columns=review_columns)
    sample = df.sample(n=min(n, len(df)), random_state=random_state).reset_index(drop=True)
    out_path = REPO_ROOT / "tests" / f"meds_core_head_{source}.csv"
    sample.to_csv(out_path, index=False)
    return out_path


# export_small_sample("mimic", n=100)
# export_small_sample("ahs", n=100)